<a href="https://colab.research.google.com/github/jacobwechuli/mypython/blob/main/wechuli_Loan_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np

# --- Load the CSV file ---
df = pd.read_csv('/bank_loan.csv')

# --- Inspect the data ---
print(df.head())
print(df.info())
print(df.shape)          # rows, columns
print(df.describe())     # summary stats for numeric columns


# Identify missing values

print(df.isnull().sum())          # count of missing values per column
print(df.isnull().mean() * 100)   # % missing per column


# Handle missing values

numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Filled missing values in '{col}' with median: {median_val}")

# 2. Categorical columns (e.g. Loan_Status, Property_Area, Employment_Type):
#    Fill with mode (most frequent value), since mean/median don't apply.
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"Filled missing values in '{col}' with mode: {mode_val}")

# 3. If a column has too much missing data (e.g. >50%), it may be better to drop it
#    rather than fill it, since imputation could introduce bias.
threshold = 0.5
cols_to_drop = df.columns[df.isnull().mean() > threshold]
if len(cols_to_drop) > 0:
    print(f"Dropping columns with >50% missing data: {list(cols_to_drop)}")
    df = df.drop(columns=cols_to_drop)


# Check for and remove duplicates

num_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows found: {num_duplicates}")

df = df.drop_duplicates()
print(f"Shape after removing duplicates: {df.shape}")


# Fix inconsistencies


# 1. Standardize text formatting (e.g. "yes"/"Yes"/"YES" should all match)
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()

# 2. Fix inconsistent category labels (example — adjust to your actual values)
# df['Loan_Status'] = df['Loan_Status'].replace({'Y': 'Yes', 'N': 'No'})

# 3. Check for and correct impossible/invalid values
#    e.g. negative income, age > 120, negative loan amounts
for col in numeric_cols:
    if col in df.columns and (df[col] < 0).any():
        print(f"Found negative values in '{col}' — review and correct these.")

# 4. Ensure numeric columns are actually numeric (sometimes numbers get read as text)
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 5. Check final data types
print(df.dtypes)


# Documentation

"""
Cleaning Summary:
- Checked for missing values using df.isnull().sum(); found missing data in
  [list columns here].
- Filled missing numeric values with the column median to avoid distortion
  from outliers.
- Filled missing categorical values with the column mode (most common value).
- Dropped [X] duplicate rows found using df.duplicated().
- Standardized text fields (trimmed whitespace, consistent capitalization)
  to avoid treating "yes" and "Yes" as different categories.
- Verified no negative values existed in numeric fields like income/loan amount.
- Confirmed all numeric columns were properly typed as numeric.
"""


# STEP 4: Save Your Cleaned Dataset
df.to_csv('bank_loan_cleaned.csv', index=False)

# Optional: download it directly to your computer
from google.colab import files
files.download('bank_loan_cleaned.csv')

   ID  Age  Experience  Income  ZIP Code  Family CCAvg  Education  Mortgage  \
0   1   25           1      49     91108       4  1/60          1         0   
1   2   45          19      34     90089       3  1/50          1         0   
2   3   39          15      11     94720       1  1/00          1         0   
3   4   35           9     100     94112       1  2/70          2         0   
4   5   35           8      45     91330       4  1/00          2         0   

   Personal Loan  Securities Account  CD Account  Online  CreditCard  
0              0                   1           0       0           0  
1              0                   1           0       0           0  
2              0                   0           0       0           0  
3              0                   0           0       0           0  
4              0                   0           0       0           1  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 co

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>